# 3 — M3C 3-φ: L0 (Médio) vs L1 (Chaveado)

> **Objetivo**: comparar o plant L0 (Venturini-style, sem ripple)
> com o L1 chaveado (9 módulos + chaves + capacitores via cost
> function + cap-outer loop). Validar que o **fundamental** do L1
> bate com a previsão analítica do L0, com ripple e harmônicos
> esperados do escalonamento multinível.

**Referências da tese**
* Sec 6 — Resultados de simulação (Tab 15)
* Tab. 16 — parâmetros do HIL/OPAL-RT
* Figs 87-98 — formas de onda em regime permanente


In [ ]:
import sys, os
from pathlib import Path
_HERE = Path.cwd() / "projects" / "inverters" / "m3c_3phase"
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

In [ ]:
from m3c_3phase_model import (
    M3cParams, build_l0_plant, build_l1_plant,
    run_l0_open_loop, run_l1_dq_full_closed_loop_with_cap_loop,
    predict_i_out_peak, predict_load_impedance, thd,
)

params = M3cParams()        # Tab. 16 defaults
print(f"Ponto de operação Tab 16:")
print(f"  Saída: {params.V_out_LL_peak/np.sqrt(2)/1000:.1f} kV LL @ {params.f_out} Hz")
print(f"  Carga: R={params.R_load} Ω, L={params.L_load*1000:.1f} mH")
print(f"  |Z|: {abs(predict_load_impedance(params)):.2f} Ω")
print(f"  Pico de corrente esperado: {predict_i_out_peak(params):.2f} A")


## 3.1 — Rodar L0 (Venturini ideal)


In [ ]:
plant_l0 = build_l0_plant(params)
res_l0 = run_l0_open_loop(plant_l0, t_end=250e-3, dt=20e-6)
print(f"L0 done. n_samples = {len(res_l0.t)}")


## 3.2 — Rodar L1 (chaveado, com closed-loop)


In [ ]:
plant_l1 = build_l1_plant(params)
i_d_ref = predict_i_out_peak(params)
res_l1, ctrl, _, _, cap_loop = run_l1_dq_full_closed_loop_with_cap_loop(
    plant_l1, params,
    i_d_in_ref=0.0,
    i_d_out_ref=i_d_ref, i_q_out_ref=0.0,
    t_end=250e-3, dt=25e-6,
)
print(f"L1 done. n_samples = {len(res_l1.t)}")
print(f"  Cap-loop correction: {cap_loop.last_correction:+.2f} A")
print(f"  V_caps mean: {np.mean(ctrl.v_caps_module):.0f} V (target {params.v_cap_total_per_module:.0f})")


## 3.3 — Comparação no domínio do tempo


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(res_l0.t*1000, res_l0.i_a_out, label="L0 (ideal)", linewidth=1.5, alpha=0.8)
axes[0].plot(res_l1.t*1000, res_l1.i_a_out, label="L1 (chaveado)", linewidth=0.8, alpha=0.7)
axes[0].set_xlim(150, 250)
axes[0].set_ylabel("i_a_out [A]")
axes[0].set_title(f"M3C — Tab 16 ({params.V_out_LL_peak/np.sqrt(2)/1000:.0f} kV / {params.f_out} Hz)")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)

axes[1].plot(res_l0.t*1000, res_l0.i_b_out, label="L0", linewidth=1.5, alpha=0.8)
axes[1].plot(res_l1.t*1000, res_l1.i_b_out, label="L1", linewidth=0.8, alpha=0.7)
axes[1].set_xlim(150, 250)
axes[1].set_xlabel("Tempo [ms]")
axes[1].set_ylabel("i_b_out [A]")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3.4 — Comparação no domínio da frequência (FFT)


In [ ]:
fs_l0 = 1.0/20e-6
fs_l1 = 1.0/25e-6
# Janela: últimos 3 períodos.
n_per_l0 = int(round((1.0/params.f_out) * fs_l0))
n_per_l1 = int(round((1.0/params.f_out) * fs_l1))

ia_l0 = res_l0.i_a_out[-3*n_per_l0:]
ia_l1 = res_l1.i_a_out[-3*n_per_l1:]

spec_l0 = np.abs(np.fft.rfft(ia_l0)) * 2.0 / len(ia_l0)
spec_l1 = np.abs(np.fft.rfft(ia_l1)) * 2.0 / len(ia_l1)
f_l0 = np.fft.rfftfreq(len(ia_l0), 1.0/fs_l0)
f_l1 = np.fft.rfftfreq(len(ia_l1), 1.0/fs_l1)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.semilogy(f_l0, spec_l0, label="L0", linewidth=1.5, alpha=0.8)
ax.semilogy(f_l1, spec_l1, label="L1", linewidth=0.8, alpha=0.7)
ax.set_xlim(0, 500)
ax.set_ylim(1e-3, None)
ax.set_xlabel("Frequência [Hz]")
ax.set_ylabel("|i_a_out(f)| [A]")
ax.set_title("Espectro de i_a — Pico em f_out, ripple do chaveamento em torno de f_sw")
ax.axvline(params.f_out, color="red", linestyle="--", alpha=0.5, label=f"f_out={params.f_out} Hz")
ax.axvline(params.f_switching, color="orange", linestyle="--", alpha=0.5, label=f"f_sw={params.f_switching} Hz")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Quantitative.
k1_l0 = int(round(params.f_out*len(ia_l0)/fs_l0))
k1_l1 = int(round(params.f_out*len(ia_l1)/fs_l1))
print(f"Fundamental L0: {spec_l0[k1_l0]:.2f} A (predito Ohm: {predict_i_out_peak(params):.2f} A)")
print(f"Fundamental L1: {spec_l1[k1_l1]:.2f} A")
print(f"THD L0: {thd(ia_l0, fs_l0, params.f_out):.2f}%")
print(f"THD L1: {thd(ia_l1, fs_l1, params.f_out):.2f}%")


## 3.5 — Resumo

* O **fundamental do L1 bate com o L0** dentro de poucos por cento.
* O THD do L1 é da ordem de **5-10%**, vs L0 que tem THD ≈ 0 (puro
  sinusóide). O ripple do L1 vem do degrau quantizado de V_cap.
* As tensões de capacitor permanecem dentro de uma faixa razoável
  (com o cap-outer loop ativo).

Próximo notebook: resposta ao degrau em corrente dq.
